In [1]:
pip install scikit-optimize

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dropout
from tensorflow.keras.regularizers import l2

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf

# scikit-optimize imports for Bayesian tuning
from skopt import BayesSearchCV
from skopt.space import Real, Integer
from sklearn.linear_model import LinearRegression

import matplotlib.pyplot as plt

In [3]:
# 2. NumPy
np.random.seed(42)

# 3. TensorFlow
tf.random.set_seed(42)

In [4]:
# Load data
df = pd.read_csv("./processed_plays.csv")

# Train/Validation/Test split (60/20/20)
train_val, test = train_test_split(df, test_size=0.20, random_state=42)
train, val      = train_test_split(train_val, test_size=0.20, random_state=42)

X_train, y_train = train.drop('yardsGained', axis=1), train['yardsGained']
X_val,   y_val   = val.drop('yardsGained', axis=1),   val['yardsGained']
X_test,  y_test  = test.drop('yardsGained', axis=1),  test['yardsGained']

In [5]:
df.head()

,down,yardsToGo,absoluteYardlineNumber,quarter,playAction,qbSneak,pff_runPassOption,yardsGained,secondsRemainingInQuarter,offFormation_EMPTY,...,passCoverage_Cover_3_Seam,passCoverage_Cover_6_Right,passCoverage_Goal_Line,passCoverage_Miscellaneous,passCoverage_Prevent,passCoverage_Quarters,passCoverage_Red_Zone,manZone_Man,manZone_Other,manZone_Zone
0,1,10,21,3,0,0,0,9,114,1,...,0,0,0,0,0,0,0,0,0,1
1,1,10,8,4,0,0,0,4,133,1,...,0,0,0,0,0,1,0,0,0,1
2,3,12,20,4,0,0,0,6,120,0,...,0,0,0,0,0,1,0,0,0,1
3,2,10,23,1,0,0,0,4,568,0,...,0,0,0,0,0,1,0,0,0,1
4,2,8,27,3,1,0,0,-1,136,0,...,0,0,0,0,0,0,0,1,0,0


In [7]:
mean_yards = y_train.mean()

y_val_baseline  = np.full_like(y_val, fill_value=mean_yards)
y_test_baseline = np.full_like(y_test, fill_value=mean_yards)

mse_val  = mean_squared_error(y_val, y_val_baseline)
mae_val  = mean_absolute_error(y_val, y_val_baseline)
rmse_val = np.sqrt(mse_val)

print(f"Baseline Model (predict mean yardage = {mean_yards:.2f}) on Validation Set:")
print(f"  MSE : {mse_val:.3f}")
print(f"  MAE : {mae_val:.3f}")
print(f"  RMSE: {rmse_val:.3f}")


Baseline Model (predict mean yardage = 5.56) on Validation Set:
  MSE : 68.722
  MAE : 5.573
  RMSE: 8.290
